# 🚀 Data Engineering Pipeline Tutorial - Part 3
## Transformation, Loading & Orchestration

### What Our Pipeline Achieved:
```
Gold Zone → PostgreSQL Warehouse:
├── mysql_customers: 5 rows
├── mysql_orders: 7 rows
├── customers: 5 rows
├── products: 5 rows
└── exchange_rates: 1 row

Total: 23 rows processed in 2.26 seconds
```

---
## 🔄 Part 6: Data Transformation (Silver → Gold)

**Gold zone = business-ready data.**

Transformations include:
- Adding audit columns (created_at, updated_at, batch_id)
- Creating surrogate keys
- Aggregations and calculations
- Joining data from multiple sources

In [ ]:
import pandas as pd
from datetime import datetime

class DataTransformer:
    """
    Transform data for Gold zone.
    Actual code from our working pipeline!
    """
    
    @staticmethod
    def add_audit_columns(df: pd.DataFrame) -> pd.DataFrame:
        """
        Add standard audit columns.
        
        Why audit columns?
        - Track when data was loaded
        - Debug data issues
        - Support incremental loads
        - Compliance requirements
        """
        df['created_at'] = datetime.now()
        df['updated_at'] = datetime.now()
        # Batch ID = timestamp, useful for identifying load batches
        df['etl_batch_id'] = datetime.now().strftime('%Y%m%d%H%M%S')
        return df
    
    @staticmethod
    def add_surrogate_key(df: pd.DataFrame, key_name: str = 'sk_id') -> pd.DataFrame:
        """
        Add surrogate key column.
        
        Surrogate key = artificial unique identifier
        Why? Natural keys can change, surrogate keys don't.
        """
        df[key_name] = range(1, len(df) + 1)
        return df

# DEMO: Transform data for Gold zone
sample_data = pd.DataFrame({
    'customer_id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie'],
    'email': ['alice@email.com', 'bob@email.com', 'charlie@email.com']
})

transformer = DataTransformer()

# Add audit columns (this is what our pipeline does)
df = transformer.add_audit_columns(sample_data.copy())
df = transformer.add_surrogate_key(df, 'dim_customer_sk')

print("✅ Transformed data with audit columns:")
print(df)

---
## 📥 Part 7: Loading to PostgreSQL Warehouse

### Our Pipeline Results:
```
PostgreSQL Warehouse (postgres:5432/devdb):
├── mysql_customers: 5 rows ✅
├── mysql_orders: 7 rows ✅
├── customers: 5 rows ✅
├── products: 5 rows ✅
└── exchange_rates: 1 row ✅
```

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
from typing import List

# Your actual PostgreSQL config
POSTGRES_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "devdb",
    "user": "devuser",
    "password": "devpassword"
}

class PostgresLoader:
    """
    Load data into PostgreSQL warehouse.
    Actual code from our working pipeline!
    """
    
    def __init__(self, config: dict):
        self.config = config
        self.conn = None
    
    def connect(self):
        """Establish connection using **kwargs."""
        self.conn = psycopg2.connect(**self.config)
        print("✅ Connected to PostgreSQL")
    
    def close(self):
        if self.conn:
            self.conn.close()
            print("✅ Connection closed")
    
    def load(self, df: pd.DataFrame, table_name: str, 
             batch_size: int = 10000) -> int:
        """
        Batch load DataFrame to PostgreSQL.
        
        Why batch insert?
        - Single INSERT per row = slow (network overhead per row)
        - Batch INSERT = fast (one network call per batch)
        
        execute_values is 10-100x faster than row-by-row inserts!
        """
        if not self.conn:
            self.connect()
        
        # Build column list with quotes (handles special chars)
        columns = ', '.join([f'"{col}"' for col in df.columns])
        
        # Convert DataFrame to list of tuples
        values = [tuple(row) for row in df.values]
        
        insert_sql = f"INSERT INTO {table_name} ({columns}) VALUES %s"
        
        rows_loaded = 0
        with self.conn.cursor() as cur:
            # Process in batches
            for i in range(0, len(values), batch_size):
                batch = values[i:i + batch_size]
                execute_values(cur, insert_sql, batch)  # Bulk insert!
                rows_loaded += len(batch)
        
        self.conn.commit()  # IMPORTANT: Save changes!
        print(f"✅ Loaded {rows_loaded} rows to {table_name}")
        return rows_loaded

# Verify our pipeline results
loader = PostgresLoader(POSTGRES_CONFIG)
loader.connect()

# Query the tables we created
with loader.conn.cursor() as cur:
    cur.execute("SELECT tablename FROM pg_tables WHERE schemaname='public'")
    tables = [t[0] for t in cur.fetchall()]
    
    print("\n📊 Tables in PostgreSQL warehouse:")
    for table in tables:
        cur.execute(f"SELECT COUNT(*) FROM {table}")
        count = cur.fetchone()[0]
        print(f"   • {table}: {count} rows")

loader.close()

In [ ]:
# Query actual data from our pipeline
import pandas as pd
import psycopg2

conn = psycopg2.connect(**POSTGRES_CONFIG)

# Read customers table
print("📋 mysql_customers table:")
df = pd.read_sql("SELECT * FROM mysql_customers LIMIT 5", conn)
print(df)

print("\n📋 mysql_orders table:")
df = pd.read_sql("SELECT * FROM mysql_orders LIMIT 5", conn)
print(df)

conn.close()

---
## 🎭 Part 8: Pipeline Orchestration

### Using Dataclasses and Enums

**Dataclass** = class for storing data, auto-generates boilerplate

**Enum** = fixed set of named constants

In [ ]:
from dataclasses import dataclass
from enum import Enum
from datetime import datetime
from typing import List

class PipelineStatus(Enum):
    """
    Enum = fixed set of named values.
    
    Better than strings:
    - Typos caught at compile time
    - IDE autocomplete works
    - Self-documenting code
    """
    PENDING = "pending"
    RUNNING = "running"
    SUCCESS = "success"   # Our pipeline achieved this! ✅
    FAILED = "failed"
    PARTIAL = "partial"   # Some steps failed

@dataclass
class PipelineResult:
    """
    Dataclass auto-generates:
    - __init__(self, status, start_time, ...)
    - __repr__ for nice printing
    - __eq__ for comparison
    
    Much cleaner than writing all that boilerplate!
    """
    status: PipelineStatus
    start_time: datetime
    end_time: datetime
    rows_processed: int
    errors: List[str]
    
    @property  # Computed property - accessed like attribute
    def duration_seconds(self) -> float:
        return (self.end_time - self.start_time).total_seconds()

# Our actual pipeline result!
result = PipelineResult(
    status=PipelineStatus.SUCCESS,
    start_time=datetime(2026, 1, 19, 21, 7, 41),
    end_time=datetime(2026, 1, 19, 21, 7, 43),
    rows_processed=23,
    errors=[]
)

print("📊 Pipeline Execution Result:")
print(f"   Status: {result.status.value}")
print(f"   Duration: {result.duration_seconds:.2f} seconds")
print(f"   Rows processed: {result.rows_processed:,}")  # :, adds commas
print(f"   Errors: {len(result.errors)}")

---
## ⏰ Part 9: Pipeline Triggers

### Ways to Start a Pipeline:
1. **Manual** - `python pipeline.py`
2. **Scheduled** - Run at specific times (like cron)
3. **Event-driven** - Trigger when new files arrive

In [ ]:
import schedule
import time
from typing import Callable

class ScheduleTrigger:
    """
    Run pipeline on a schedule.
    
    Callable = type hint meaning "any function"
    """
    
    def __init__(self, pipeline_func: Callable):
        self.pipeline_func = pipeline_func
    
    def run_daily(self, at_time: str = "02:00"):
        """
        Run pipeline daily at specified time.
        
        Why 2 AM? 
        - Low system usage
        - Data from previous day is complete
        - Results ready for morning reports
        """
        schedule.every().day.at(at_time).do(self.pipeline_func)
        print(f"📅 Scheduled daily run at {at_time}")
    
    def run_every(self, minutes: int):
        """Run pipeline every N minutes."""
        schedule.every(minutes).minutes.do(self.pipeline_func)
        print(f"📅 Scheduled run every {minutes} minutes")

# Example usage (don't run in notebook - it blocks!)
def my_pipeline():
    print(f"🚀 Pipeline running at {datetime.now()}")
    # ... pipeline code ...
    return True

# How to use:
# trigger = ScheduleTrigger(my_pipeline)
# trigger.run_daily("02:00")  # Run at 2 AM daily

print("To run scheduled pipeline:")
print("  python triggers.py schedule")

---
## 🔔 Part 10: Logging & Notifications

**Always log in production!** Logs are your debugging lifeline.

In [ ]:
import logging

# Setup logging - this is what our pipeline uses
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('pipeline.log'),  # Write to file
        logging.StreamHandler()                # Also print to console
    ]
)
logger = logging.getLogger(__name__)

# Different log levels
logger.debug("Detailed info for debugging")    # Usually hidden
logger.info("General information")             # Normal operations
logger.warning("Something unexpected")         # Not an error, but notable
logger.error("Something failed")               # Errors that need attention
logger.critical("System is down!")             # Severe errors

print("\n📄 Check pipeline.log for full logs")

---
## 🏆 Complete Pipeline Summary

### What We Built:

```
┌─────────────────────────────────────────────────────────────────┐
│                     DATA SOURCES                                 │
│  MySQL (customers, orders) + CSV (2 files) + API (exchange)     │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│  BRONZE ZONE (s3://data-lake/bronze/)         │
│  Raw data: parquet, csv, json formats                           │
│  5 files uploaded                                                │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│  SILVER ZONE (s3://data-lake/silver/)         │
│  Cleaned: standardized columns, removed duplicates, typed       │
│  5 parquet files                                                 │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│  GOLD ZONE (s3://data-lake/gold/)             │
│  Transformed: audit columns added                                │
│  5 parquet files                                                 │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│  POSTGRESQL WAREHOUSE (postgres:5432/devdb)                    │
│  Tables: mysql_customers, mysql_orders, customers, products,    │
│          exchange_rates                                          │
│  Total: 23 rows loaded                                           │
└─────────────────────────────────────────────────────────────────┘
```

### Python Patterns Learned:

| Pattern | Syntax | Use Case |
|---------|--------|----------|
| Decorator | `@retry_on_failure()` | Retry logic |
| Generator | `yield batch` | Memory-efficient processing |
| Dataclass | `@dataclass` | Structured results |
| Enum | `class Status(Enum)` | Type-safe constants |
| Lambda | `lambda x: x > 0` | Inline validation |
| Type hints | `def f(x: str) -> bool` | Documentation |
| **kwargs | `func(**config)` | Flexible parameters |
| Context manager | `with conn:` | Resource cleanup |

### Commands to Run:

```bash
cd tina-data-engineer/workspace/data_pipeline

# Validate setup
python3 validate.py

# Run pipeline
python3 pipeline.py

# Check logs
cat pipeline.log
```